In [3]:
import requests
import html

AUTHOR_RECID = "1046872"
OUTPUT_FILE = "publications.html"

def fetch_all_papers():
    url = "https://inspirehep.net/api/literature"
    papers = []
    page = 1
    size = 100

    while True:
        params = {
            "q": f"authors.recid:{AUTHOR_RECID}",
            "size": size,
            "page": page,
            "sort": "mostrecent"
        }

        res = requests.get(url, params=params)
        data = res.json()

        hits = data.get("hits", {}).get("hits", [])
        if not hits:
            break

        papers.extend(hits)
        print(f"Fetched page {page} ({len(hits)} records)")
        page += 1

    return papers


def extract_info(hit):
    metadata = hit.get("metadata", {})

    # -------------------
    # Authors
    # -------------------
    authors = metadata.get("authors", [])
    author_names = ", ".join(a.get("full_name", "") for a in authors)

    # -------------------
    # Title
    # -------------------
    title = metadata.get("titles", [{}])[0].get("title", "No title")

    # -------------------
    # Year + Journal (FIXED)
    # -------------------
    journal = ""
    year = ""

    pub_info_list = metadata.get("publication_info", [])

    if pub_info_list:
        pub_info = pub_info_list[0]

        journal = pub_info.get("journal_title", "") or ""
        year = pub_info.get("year", "") or ""

        # Sometimes year is missing → fallback
        if not year:
            year = metadata.get("preprint_date", "")[:4]

    # -------------------
    # arXiv link (FIXED)
    # -------------------
    arxiv_link = "#"

    # Primary method
    arxiv_eprints = metadata.get("arxiv_eprints", [])
    if arxiv_eprints:
        arxiv_id = arxiv_eprints[0].get("value")
        if arxiv_id:
            arxiv_link = f"https://arxiv.org/abs/{arxiv_id}"

    # Fallback method
    if arxiv_link == "#":
        for eid in metadata.get("external_system_identifiers", []):
            if eid.get("schema") == "arxiv":
                arxiv_link = f"https://arxiv.org/abs/{eid.get('value')}"
                break

    # -------------------
    # DOI (more robust)
    # -------------------
    doi_link = "#"
    dois = metadata.get("dois", [])

    if dois:
        doi_value = dois[0].get("value")
        if doi_value:
            doi_link = f"https://doi.org/{doi_value}"

    return author_names, title, journal, year, arxiv_link, doi_link


def generate_html(papers):
    html_items = []

    for hit in papers:
        authors, title, journal, year, arxiv, doi = extract_info(hit)

        # Better formatting
        journal_text = journal if journal else "Preprint"

        item = f"""
<li>
  <b>{html.escape(authors)}</b>, "{html.escape(title)}", {html.escape(journal_text)} ({year})  
  <a href="{arxiv}">[arXiv]</a> | <a href="{doi}">[DOI]</a>
</li>
"""
        html_items.append(item.strip())

    return "\n".join(html_items)


def main():
    papers = fetch_all_papers()
    print(f"\nTotal papers fetched: {len(papers)}")

    html_list = generate_html(papers)

    full_html = f"""
{html_list}
"""

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(full_html)

    print(f"\nHTML file saved as: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Fetched page 1 (55 records)

Total papers fetched: 55

HTML file saved as: publications.html
